In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [3]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = LANGCHAIN_PROJECT

In [4]:
LANGCHAIN_PROJECT

'langgraph-prerequist'

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI


embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [42]:
while True:
    question = input(
        "Type your question. If you want to quit, write quit: "
    )

    if question.lower() == "quit":
        print("Goodbye, take care!")
        break

    try:
        response = llm.invoke(question)
        print(response.text)

    except Exception as e:
        print(f"Error calling Gemini: {e}")

Hello! How can I help you today?
**LangGraph** is an open-source framework developed by the team behind **LangChain**. It is designed specifically for building complex, **stateful, multi-actor LLM applications** using graph-based orchestration.

While traditional LangChain is great for building linear "chains" of LLM calls, LangGraph is built to support **cycles, loops, memory, and human intervention**, which are required to build truly autonomous AI agents.

---

### Why was LangGraph created? (The Problem it Solves)

Standard LLM chains are **DAGs** (Directed Acyclic Graphs). They move strictly in one direction: 
`Input → Prompt → LLM → Output`

However, real-world AI agents don't work linearly. They need to:
1. **Loop back:** Try a task, evaluate the result, and try again if it failed (e.g., self-correcting code generation).
2. **Maintain State:** Remember what happened 5 steps ago.
3. **Wait for Humans:** Pause execution, ask a human for input/approval, and resume.
4. **Coordinate 

In [43]:
while True:
    question = input(
        "Type your question. If you want to quit, write quit: "
    )

    if question.lower() == "quit":
        print("Goodbye, take care!")
        break

    try:
        response = llm.invoke(question)
        print(response.text)

    except Exception as e:
        print(f"Error calling Gemini: {e}")

Hello Joe! It's nice to meet you. How can I help you today?
I don't know your name yet, as I don't have access to your personal information! What would you like me to call you?
Goodbye, take care!


In [44]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage 

In [45]:
store={}

In [46]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [47]:
config = {"configurable": {"session_id": "firstchat"}}

In [49]:
model_with_memory = RunnableWithMessageHistory(llm, get_session_history)

In [51]:
model_with_memory.invoke("Hello, I'm Joe Wilson Thamiyan?", config=config).text

'Hello again, Joe! Yes, I received your introduction. \n\nSince you included a question mark, are you asking if I recognize your name/have information about you, or did you just want to say hello? \n\nLet me know how I can help you today!'

In [52]:
model_with_memory.invoke("Tell me what is my name?", config=config).text

'Based on what you introduced yourself as, your name is **Joe Wilson Thamiyan**!'

In [76]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate   
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [60]:
# Reading the txt files from the directory
loader = DirectoryLoader('./data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

In [ ]:
# Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, 
    chunk_overlap=0, 
    length_function=len
)

new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

In [72]:
# Creating Retriever using Chroma Vector Store
db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [75]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [77]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)

In [83]:
question ="How do I calculate annual crop turns??"
print(retrieval_chain.invoke(question))

Based on the provided context, crop turns are calculated as **365 divided** (the provided text cuts off and does not specify the rest of the formula).


### Tools and Agents

In [104]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper 

In [105]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=100)

In [106]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [107]:
tool.name

'wikipedia'

In [108]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [109]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [102]:
tool.return_direct

False

In [ ]:
print(tool.invoke({"query": "langGraph"}))

In [115]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=4000
)

tool = WikipediaQueryRun(
    api_wrapper=api_wrapper
)

print("Name:", tool.name)
print("Description:", tool.description)
print("Args:", tool.args)

try:
    result = tool.invoke({
        "query": "LangGraph"
    })

    print("\nResult:")
    print(result)

except Exception as e:
    print("\nError:")
    print(type(e).__name__, e)

Name: wikipedia
Description: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
Args: {'query': {'description': 'query to look up on wikipedia', 'title': 'Query', 'type': 'string'}}

Error:
JSONDecodeError Expecting value: line 1 column 1 (char 0)


### Tool: YouTube-Search

In [116]:
from langchain_community.tools import YouTubeSearchTool

In [117]:
tool=YouTubeSearchTool()

In [118]:
tool.name

'youtube_search'

In [119]:
tool.description

'search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional'

In [121]:
tool.run("sunny savita")

"['https://www.youtube.com/watch?v=s7SZoj0kLAU&pp=ygUMc3Vubnkgc2F2aXRh', 'https://www.youtube.com/watch?v=EyXzfnAxCdA&pp=ygUMc3Vubnkgc2F2aXRh']"

In [124]:
from langchain_tavily import TavilySearch

In [128]:
tool_3 =TavilySearch(max_results=5)

In [129]:
result = tool_3.invoke({
    "query": "latest Nepal floods news September 2026 what happened"
})

print(result)

{'query': 'latest Nepal floods news September 2026 what happened', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/2026_Nepal%E2%80%93Tibet_floods', 'title': '2026 Nepal–Tibet floods - Wikipedia', 'content': '4. 1 2 3 4 5 "西藏吉隆泥石流灾害已致31人遇难531人失联" [The mudslide disaster in Gyirong, Tibet has left 31 people dead and 531 missing (as of 2026-09-02)]. 新华社 (Xinhua News Agency). 4 September 2026. Retrieved 4 September 2026.\n5. 1 2 "Nepal-China flash floods death toll rises to 1,280, over 5,620 missing as search, rescue enter 2nd week". Anadolu Agency. Retrieved 3 September 2026. [...] 82. 1 2 Paudyal, Rishiram (1 September 2026). "Hundreds missing, billions in goods feared lost in Bhotekoshi flood". The Kathmandu Post. Retrieved 1 September 2026.\n83. 1 2 Kafle, Shristi; Saaliq, Sheikh (1 September 2026). "Mass burials in Nepal as death toll from flash floods exceeds 1,000". AP News. Retrieved 2 September 2026. [...] 64. ↑ Lapham,

In [137]:
from langchain.agents import create_agent

In [141]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

load_dotenv()

# LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

# Tool
# This tool performs live web search for current news.
tavily_tool = TavilySearch(max_results=5)

# Agent
agent = create_agent(
    model=llm,
    tools=[tavily_tool],
)

# Invoke agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened in the latest Nepal floods?"
        }
    ]
})

print(result["messages"][-1].content)


**What happened in the latest Nepal floods?**  
On **26 August 2026**, a massive flash‑flood struck northern Nepal, primarily affecting the districts of **Rasuwa, Nuwakot, Dhading, Gorkha, and Tanahun** along the **Bhote Koshi** and **Trishuli** river corridors. The disaster was triggered when a large chunk of a glacier on the Langtang Lirung peak in Langtang National Park collapsed, temporarily damming the river. The sudden release of meltwater and debris sent a torrent downstream, sweeping away homes, roads, bridges, and even hydropower infrastructure.

| What happened | Key facts |
|---------------|-----------|
| **Cause** | Glacier collapse and meltwater surge (likely a large ice‑rock mass breaking off)【1†L1-L6】 |
| **Affected areas** | Rasuwa, Nuwakot, Dhading, Gorkha, Tanahun (and downstream along Trishuli)【1†L1-L6】【2†L1-L5】 |
| **Casualties** | At least **579** confirmed deaths (toll rising) and **nearly 2,000** missing as of 28 Aug 2026【3†L1-L6】 |
| **Displacement** | Tens of t

In [20]:
# Invoke agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the latest developments in India's AI sector?"
        }
    ]
})

print(result["messages"][-1].content)

**Latest developments in India’s AI sector (as of September 2026)**  

| Area | Key developments | Source |
|------|------------------|--------|
| **Talent & hiring** | AI & machine‑learning roles drove a 14 % YoY jump in white‑collar hiring in August 2026, with firms expanding local talent pipelines and investing in STEM education. | [Staffing Industry](https://www.staffingindustry.com/news/global-daily-news/indias-ai-hiring-surge-fuels-14-jump-in-white-collar-recruitment) |
| **Investment & spend** | AI investment by Indian companies rose 119 % YoY in 2026, reflecting a sharp uptick in spend on data‑center infrastructure and AI‑enabled services. | [IndianBlog.co.in](https://indianblog.co.in/india-and-ai-in-2026-how-india-is-building-its-artificial-intelligence-future) |
| **Governance & security** | Industry leaders called for stronger AI governance to counter the expanding cyber‑threat landscape posed by autonomous agents. Emphasis on “security‑by‑design” and aligning security budge